# unialg — Composing Programs with the DSL

This notebook introduces the unialg DSL from the ground up. You'll learn how to write, compile, and run programs using the surface syntax — the same text language used throughout the project.

Programs in unialg are built from **morphisms**: typed transformations that compose. You write them as text, compile them with `compile_program()`, and run them with `.run()`. No internal imports needed.

We'll start with the simplest possible programs and progressively build toward recursive types, recursion schemes, and tensor contraction over custom semirings.

In [1]:
import sys
sys.path.insert(0, "/home/scanbot/unialg/src")

import numpy as np
from unialg import compile_program

## 1. Hello World — Your First Program

A program is a string of declarations. The simplest useful program loads a backend (numpy) and names a morphism with `let`.

`compile_program()` parses the text, type-checks the composition, and returns a compiled program. `.run()` executes it.

In [2]:
prog = compile_program("load numpy\nlet f = tanh")

result = prog.run(np.array([0.0, 1.0, -1.0]))
print("tanh:", result)

tanh: [ 0.          0.76159416 -0.76159416]


That's the entire pipeline: text → compile → run → result.

`load numpy` brings the numpy backend's operations into scope. `let f = tanh` names a morphism that applies the tanh function. The last `let` is what gets compiled.

In [3]:
prog = compile_program("load numpy\nlet f = exp")
print("exp:", prog.run(np.array([0.0, 1.0, 2.0])))

prog = compile_program("load numpy\nlet f = sqrt")
print("sqrt:", prog.run(np.array([4.0, 9.0, 16.0])))

exp: [1.         2.71828183 7.3890561 ]
sqrt: [2. 3. 4.]


## 2. Composition — Chaining Transformations

The `>>` operator composes morphisms left to right (diagrammatic order). `f >> g` means "apply f, then g".

In [4]:
# exp then log should be the identity (roundtrip)
prog = compile_program("load numpy\nlet f = exp >> log")

x = np.array([1.0, 2.0, 3.0])
result = prog.run(x)
print("exp >> log:", result)
print("original: ", x)

exp >> log: [1. 2. 3.]
original:  [1. 2. 3.]


In [5]:
# square each element, then take the square root
prog = compile_program("load numpy\nlet f = multiply >> sqrt")

x = np.array([4.0, 9.0])
result = prog.run(x, x)  # multiply takes two inputs
print("multiply >> sqrt:", result)

multiply >> sqrt: [4. 9.]


## 3. Binary Operations and Products

Some morphisms take two inputs — they consume a **product type** (a pair). Pass both to `.run()`.

In [6]:
prog = compile_program("load numpy\nlet f = add")

a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])
print("add:", prog.run(a, b))

add: [11. 22. 33.]


In [7]:
prog = compile_program("load numpy\nlet f = multiply")
print("multiply:", prog.run(np.array([2.0, 3.0]), np.array([4.0, 5.0])))

multiply: [ 8. 15.]


In [8]:
# compose a binary op with a unary op: add two arrays, then apply tanh
prog = compile_program("load numpy\nlet f = add >> tanh")

result = prog.run(np.array([0.5, 0.5]), np.array([0.5, 0.5]))
print("add >> tanh:", result)
print("expected:  ", np.tanh([1.0, 1.0]))

add >> tanh: [0.76159416 0.76159416]
expected:   [0.76159416 0.76159416]


### Pair, Parallel, Copy — Product Combinators

Products are the fundamental way to handle multiple values. Three combinators build on them:

- `f & g` (**pair**): apply both f and g to the same input, producing a pair of results
- `f || g` (**parallel**): apply f to the left input and g to the right, independently
- `copy` (**diagonal**): duplicate a single input into a pair

In [9]:
# pair: apply tanh AND exp to the same input
prog = compile_program("load numpy\nlet f = tanh & exp")
result = prog.run(np.array([1.0]))
print("tanh & exp:", result)
print("  left  (tanh):", result[0])
print("  right (exp): ", result[1])

tanh & exp: (array([0.76159416]), array([2.71828183]))
  left  (tanh): [0.76159416]
  right (exp):  [2.71828183]


In [10]:
# parallel: apply tanh to the left, exp to the right, independently
prog = compile_program("load numpy\nlet f = tanh || exp")
result = prog.run(np.array([1.0]), np.array([2.0]))
print("tanh || exp:", result)
print("  left  (tanh(1.0)):", result[0])
print("  right (exp(2.0)): ", result[1])

tanh || exp: (array([0.76159416]), array([7.3890561]))
  left  (tanh(1.0)): [0.76159416]
  right (exp(2.0)):  [7.3890561]


In [11]:
# copy: duplicate input, then add the pair — this doubles every element
prog = compile_program("load numpy\nlet double = copy >> add")
result = prog.run(np.array([3.0, 5.0, 7.0]))
print("copy >> add (double):", result)

copy >> add (double): [ 6. 10. 14.]


### Residual Connections

These combinators compose into real architectural patterns. A **residual connection** copies the input, transforms one copy, and adds it back:

```
copy >> (f || id) >> add
```

This is `x + f(x)` — the skip connection from ResNets.

In [12]:
prog = compile_program("load numpy\nlet residual = copy >> (tanh || id) >> add")

x = np.array([1.0, 2.0, 3.0])
result = prog.run(x)
print("residual (x + tanh(x)):", result)
print("expected:              ", x + np.tanh(x))

residual (x + tanh(x)): [1.76159416 2.96402758 3.99505475]
expected:               [1.76159416 2.96402758 3.99505475]


In [13]:
# parametric residual: the transformation is a parameter
src = """
load numpy
let residual(f) = copy >> (f || id) >> add
let res_tanh = residual(tanh)
let res_exp  = residual(exp)
"""

x = np.array([0.5])
for name in ["res_tanh", "res_exp"]:
    prog = compile_program(src, target=name)
    print(f"{name}({x}): {prog.run(x)}")

res_tanh([0.5]): [0.96211716]
res_exp([0.5]): [2.14872127]


## 4. Naming and Reuse — Let Bindings

Programs can define multiple morphisms. Each `let` can reference earlier names. By default, the last `let` is what gets compiled — or use `target="name"` to choose.

In [14]:
src = """
load numpy
let base = tanh
let pipeline = base >> exp
"""

prog = compile_program(src)
result = prog.run(np.array([1.0]))
print("tanh >> exp:", result)
print("expected:  ", np.exp(np.tanh([1.0])))

tanh >> exp: [2.14168768]
expected:   [2.14168768]


In [15]:
# use target= to select a specific morphism from the program
src = """
load numpy
let step1 = exp
let step2 = step1 >> tanh
let step3 = step2 >> log
"""

for name in ["step1", "step2", "step3"]:
    prog = compile_program(src, target=name)
    result = prog.run(np.array([1.0]))
    print(f"{name}: {result}")

step1: [2.71828183]
step2: [0.99132892]
step3: [-0.0087089]


## 5. Parametric Morphisms — Higher-Order Composition

Parameters in `let f(w) = ...` are **compile-time bindings**. They receive other morphisms, not runtime data. This lets you build reusable templates.

In [16]:
# f is a template: "apply w, then tanh"
# g instantiates it with exp
src = """
load numpy
let f(w) = w >> tanh
let g = f(exp)
"""

prog = compile_program(src)
result = prog.run(np.array([1.0, 2.0]))
print("f(exp) = exp >> tanh:", result)
print("expected:           ", np.tanh(np.exp([1.0, 2.0])))

f(exp) = exp >> tanh: [0.99132892 0.99999924]
expected:            [0.99132892 0.99999924]


In [17]:
# multiple parameters: f(a, b) composes a then b
src = """
load numpy
let f(a, b) = a >> b
let g = f(exp, tanh)
"""

prog = compile_program(src)
result = prog.run(np.array([1.0, 2.0]))
print("f(exp, tanh) = exp >> tanh:", result)
print("expected:                  ", np.tanh(np.exp([1.0, 2.0])))

f(exp, tanh) = exp >> tanh: [0.99132892 0.99999924]
expected:                   [0.99132892 0.99999924]


In [18]:
# nesting: g wraps f with an extra stage
src = """
load numpy
let f(w) = w >> tanh
let g(v) = f(v) >> exp
let h = g(log)
"""

prog = compile_program(src)
result = prog.run(np.array([1.0, 2.0]))
print("g(log) = log >> tanh >> exp:", result)
print("expected:                   ", np.exp(np.tanh(np.log([1.0, 2.0]))))

g(log) = log >> tanh >> exp: [1.        1.8221188]
expected:                    [1.        1.8221188]


Note: parametric `let` declarations (like `f` and `g` above) don't appear as runnable morphisms themselves — only their instantiations do. Parameters are bound at compile time, not at runtime.

## 6. Pure Structure — No Backend Needed

Not all programs need a backend. Structural morphisms — identity, injections, projections, copy, delete — work on algebraic types without any `load` directive.

These programs operate on symbolic values: products (pairs) and sums (tagged choices).

In [19]:
import hydra.dsl.meta.phantoms as P

# identity on the unit type
prog = compile_program("let f = id")
result = prog.run(P.unit().value)
print("id(unit):", result)

id(unit): None


Structural programs become interesting when we start building data out of algebraic types. The next section does exactly that.

## 7. Recursive Types — Shapes and Fixed Points

The `shape` keyword declares polynomial functors — type constructors that describe the "shape" of data.

- `1` is the unit type (one value, no information)
- `x` is the recursive variable ("put another layer here")
- `|` is sum (tagged choice: this OR that)
- `&` is product (pair: this AND that)

`fix` ties the recursive knot, creating a type that contains copies of itself.

In [20]:
# Peano natural numbers:
# NatF(X) = 1 | X      -- either Zero (left) or Successor of X (right)
# Nat = fix NatF        -- a natural number is zero or successor of another natural number

src = """
shape NatF = 1 | x
shape Nat = fix NatF

let zero = |0 >> roll[Nat]
let succ = |1 >> roll[Nat]

let one = zero >> succ
let two = one >> succ
let three = two >> succ
"""

for name in ["zero", "one", "two", "three"]:
    prog = compile_program(src, target=name)
    print(f"{name}: {prog.run()}")

zero: ('left', None)
one: ('right', ('left', None))
two: ('right', ('right', ('left', None)))
three: ('right', ('right', ('right', ('left', None))))


Each number is a nested structure of tagged values:
- `zero` = `('left', None)` — the left injection (zero case)
- `one` = `('right', ('left', None))` — successor of zero
- `two` = `('right', ('right', ('left', None)))` — successor of successor of zero

The building blocks:
- `|0` injects into the left (zero case)
- `|1` injects into the right (successor case)  
- `roll[Nat]` wraps the result as a `Nat` value

These are pure structural programs — no backend, no arrays. Just algebra.

## 8. Recursion Schemes — Fold, Unfold, and Hylo

Once you have a recursive type, recursion schemes let you process it systematically.

### Catamorphism (fold)

`cata[Nat](algebra)` folds a recursive structure down to a value. The algebra says what to do at each layer:
- For `NatF = 1 | x`, the algebra is `zero_case | successor_case`

In [21]:
def peano_to_int(value):
    """Decode a Peano-encoded number to a Python int."""
    n = 0
    while True:
        tag, payload = value
        if tag == "left":
            return n
        n += 1
        value = payload


src = """
shape NatF = 1 | x
shape Nat = fix NatF

let zero = |0 >> roll[Nat]
let succ = |1 >> roll[Nat]

let one = zero >> succ
let two = one >> succ
let three = two >> succ

let add(n) = cata[Nat](n | succ)
let mul(n) = cata[Nat](zero | add(n))

let five = three >> add(two)
let six = three >> mul(two)
"""

for name in ["five", "six"]:
    prog = compile_program(src, target=name)
    raw = prog.run()
    print(f"{name}: {peano_to_int(raw)}")

five: 5
six: 6


How `add(n)` works as a catamorphism:
- `cata[Nat](n | succ)` folds over a `Nat` value
- At each layer: if zero → return `n`; if successor → apply `succ`
- So `three >> add(two)` unfolds `three` and replaces its zero with `two`, giving five

Similarly, `mul(n)` replaces zero with `zero` (absorbing element) and successor with `add(n)`.

### Anamorphism (unfold) and Hylomorphism (unfold then fold)

`ana[Nat](coalgebra)` builds recursive structure from a seed.

`hylo[Nat](coalgebra, algebra)` unfolds then immediately folds — no intermediate structure.

In [22]:
src = """
shape NatF = 1 | x
shape Nat = fix NatF

let zero = |0 >> roll[Nat]
let succ = |1 >> roll[Nat]

let one = zero >> succ
let two = one >> succ
let three = two >> succ

let rebuild = ana[Nat](unroll[Nat])
let fold_id = cata[Nat](zero | succ)
let hylo_id = hylo[Nat](unroll[Nat], zero | succ)

let rebuilt_three = three >> rebuild
let folded_three = three >> fold_id
let hylo_three = three >> hylo_id
"""

for name in ["rebuilt_three", "folded_three", "hylo_three"]:
    prog = compile_program(src, target=name)
    print(f"{name}: {peano_to_int(prog.run())}")

rebuilt_three: 3
folded_three: 3
hylo_three: 3


All three produce 3:
- `rebuild` (ana): unfolds `three` using its own coalgebra (`unroll[Nat]`), then rebuilds it — identity
- `fold_id` (cata): folds `three` with `zero | succ` — also identity
- `hylo_id` (hylo): does both in one pass

## 9. Monadic Effects

`pure[Maybe](f)` lifts a morphism into the Maybe monad: the result is wrapped in a `Just` (success). This lets you compose morphisms that might fail with ones that always succeed.

In [23]:
src = """
shape NatF = 1 | x
shape Nat = fix NatF

let zero = |0 >> roll[Nat]
let succ = |1 >> roll[Nat]
let one = zero >> succ
let two = one >> succ
let three = two >> succ

let safe_id = cata[Nat](pure[Maybe](zero | succ))
let maybe_three = three >> safe_id
"""

prog = compile_program(src, target="maybe_three")
raw = prog.run()
print("maybe_three (raw):", raw)
print("decoded:          ", peano_to_int(raw))

maybe_three (raw): ('right', ('right', ('right', ('left', None))))
decoded:           3


The algebra `pure[Maybe](zero | succ)` wraps each recursive step in `Maybe` — all succeed, so the final result is the same number. This pattern becomes useful when some algebra steps can legitimately fail.

## 10. Tensors — Semiring Algebras and Contraction

This is where the algebraic framework meets numerical computation.

A **semiring** is a pair of operations ($\oplus$, $\otimes$) with identities, satisfying associativity, distributivity, and annihilation. The tensor equation

$$C_{ik} = \bigoplus_j \; A_{ij} \otimes B_{jk}$$

is the same structure regardless of which semiring you choose. Only the meaning changes.

Tensors are an extension — `load extension tensors` activates the `algebra` and `contract` keywords.

### Standard matrix multiplication (arithmetic semiring)

With $\oplus = +$ and $\otimes = \times$, this is ordinary matrix multiply.

In [24]:
src = """
load numpy
load extension tensors
algebra real(plus=add, times=multiply, zero=0.0, one=1.0)
let matmul = contract[real]("ij,jk->ik")
"""

prog = compile_program(src)

A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
B = np.array([[7.0, 8.0],
              [9.0, 10.0],
              [11.0, 12.0]])

result = prog.run(A, B)
print("contract[real] matmul:")
print(result)
print("\nnumpy A @ B:")
print(A @ B)
print("\nmatch:", np.allclose(result, A @ B))

contract[real] matmul:
[[ 58.  64.]
 [139. 154.]]

numpy A @ B:
[[ 58.  64.]
 [139. 154.]]

match: True


The `algebra` declaration names a semiring by binding its operations to backend primitives:
- `plus=add` → $\oplus$ is numpy addition
- `times=multiply` → $\otimes$ is numpy multiplication  
- `zero=0.0` → $\oplus$-identity (and $\otimes$-annihilator)
- `one=1.0` → $\otimes$-identity

`contract[real]("ij,jk->ik")` uses einsum-style index notation to specify which dimensions are contracted.

### Tropical semiring — shortest paths

Now swap the semiring: $\oplus = \min$, $\otimes = +$, zero $= \infty$, one $= 0$.

The same contraction equation now computes shortest paths:

$$C_{ik} = \min_j \; (A_{ij} + B_{jk})$$

Each entry $A_{ij}$ is a distance from $i$ to $j$. Adding gives total path cost; $\min$ picks the cheapest route.

In [25]:
src = """
load numpy
load extension tensors
algebra tropical(plus=minimum, times=add, zero=inf, one=0.0)
let shortest = contract[tropical]("ij,j->i")
"""

prog = compile_program(src)

# distance matrix: cost from node i to intermediate j
distances = np.array([[0.0, 4.0, 2.0],
                      [5.0, 1.0, 3.0]])
# extra cost at each intermediate node
costs = np.array([7.0, 2.0, 6.0])

result = prog.run(distances, costs)
expected = np.min(distances + costs[None, :], axis=1)

print("tropical contraction (shortest paths):")
print(result)
print("\nexpected (manual min-plus):")
print(expected)
print("\nmatch:", np.allclose(result, expected))

tropical contraction (shortest paths):
[6. 3.]

expected (manual min-plus):
[6. 3.]

match: True


In [26]:
# tropical matrix-matrix: C[i,k] = min_j(A[i,j] + B[j,k])
src = """
load numpy
load extension tensors
algebra tropical(plus=minimum, times=add, zero=inf, one=0.0)
let tmatmul = contract[tropical]("ij,jk->ik")
"""

prog = compile_program(src)

A = np.array([[0.0, 3.0],
              [2.0, 1.0]])
B = np.array([[1.0, 4.0],
              [2.0, 0.0]])

result = prog.run(A, B)
expected = np.min(A[:, :, None] + B[None, :, :], axis=1)

print("tropical matmul (min-plus):")
print(result)
print("\nexpected:")
print(expected)
print("\nmatch:", np.allclose(result, expected))

tropical matmul (min-plus):
[[1. 3.]
 [3. 1.]]

expected:
[[1. 3.]
 [3. 1.]]

match: True


In [27]:
# tropical inner product: min_i(a[i] + b[i])
src = """
load numpy
load extension tensors
algebra tropical(plus=minimum, times=add, zero=inf, one=0.0)
let dot = contract[tropical]("i,i->")
"""

prog = compile_program(src)

a = np.array([3.0, 1.0, 4.0, 1.0, 5.0])
b = np.array([2.0, 7.0, 1.0, 8.0, 2.0])

result = prog.run(a, b)
expected = np.min(a + b)

print(f"tropical inner product: {result}")
print(f"expected (min of a+b): {expected}")
print(f"match: {np.allclose(result, expected)}")

tropical inner product: 5.0
expected (min of a+b): 5.0
match: True


### The key insight

The contraction equation

$$C_{ik} = \bigoplus_j \; A_{ij} \otimes B_{jk}$$

is identical in every case. Only the semiring changes:

| Semiring | $\oplus$ | $\otimes$ | What it computes |
|----------|----------|-----------|------------------|
| Arithmetic | $+$ | $\times$ | weighted sum of paths (linear algebra) |
| Tropical | $\min$ | $+$ | cheapest path (shortest-path algorithms) |
| Fuzzy | $\max$ | $\min$ | strongest chain of evidence |

Any computation expressible as a tensor contraction inherits this freedom: change the semiring, change the meaning, keep the structure.

## 11. Putting It Together — A GPT-2 Transformer Block

A single GPT-2 transformer block, built entirely from morphism combinators and semiring contractions.

The algebraic decomposition follows Gavranović et al. (2024): neural networks are **algebra homomorphisms**
for endofunctors, valued in the 2-category **Para** of parametric maps.

**Tensor primitives** — semiring contractions reused with different weight matrices:
- `proj = contract[real]("ij,j->i")` — matrix-vector projection
- `hadamard = contract[real]("i,i->i")` — element-wise product

**Attention** — a single composed morphism via structural combinators:
- `copy >> (copy || id)` fans input x into three copies ((x,x),x)
- `(q || k) || v` applies three projections in parallel
- `score_and_gate || id` scores and softmax-gates, leaving V untouched
- final `hadamard` mixes gated weights with values

**Residual** — the comonoid pattern `copy >> (f || id) >> add` gives x + f(x).

**Layer stacking** — a catamorphism over the list functor `1 | x`. Each fold step
applies one transformer block. For weight-tied models, the same block algebra is
reused at every layer.

Each call to `proj` is a parametric morphism (P, A) → B in **Para** — the same
contraction instantiated with different parameter tensors.

In [ ]:
import sys
sys.setrecursionlimit(10000)
from scipy.special import softmax as scipy_softmax

gpt2_src = """
load numpy
load extension tensors
algebra real(plus=add, times=multiply, zero=0.0, one=1.0)

# Tensor primitives: semiring contractions
let proj = contract[real]("ij,j->i")
let hadamard = contract[real]("i,i->i")
let soft = softmax(x, '-1')

# Scoring: element-wise product then softmax
let score_and_gate = hadamard >> soft

# Residual: comonoid copy, transform one branch, add back
let residual(f) = copy >> (f || id) >> add

# Single-head attention as one composed morphism:
#   fan x three ways, project each, score Q*K, softmax, mix with V
let attention(q, k, v) = copy >> (copy || id) >> ((q || k) || v) >> (score_and_gate || id) >> hadamard

# FFN: up-project, activate, down-project
let ffn(up, down) = up >> tanh >> down

# Full transformer block: attention + residual, then FFN + residual
let attn_head = attention(proj, proj, proj)
let ffn_layer = ffn(proj, proj)
let block = residual(attn_head) >> residual(ffn_layer)

# Layer stacking: catamorphism over list functor
shape LayerF = 1 | x
shape Layers = fix LayerF
let stack = cata[Layers](id | block)
"""

# --- Compile and run a single block ---
prog = compile_program(gpt2_src, target='block')

d = 4
np.random.seed(42)
Wq = np.random.randn(d, d) * 0.1
Wk = np.random.randn(d, d) * 0.1
Wv = np.random.randn(d, d) * 0.1
W_up = np.random.randn(d, d) * 0.1
W_down = np.random.randn(d, d) * 0.1
x = np.random.randn(d)

print("Input:", x)
result = prog.run(Wq, Wk, Wv, W_up, W_down, x)
print("Block output:", result)

# --- Oracle verification ---
q = Wq @ x; k = Wk @ x; v = Wv @ x
w = scipy_softmax(q * k, axis=-1)
ctx = w * v
h = ctx + x  # attention residual
h_up = np.tanh(W_up @ h)
h_down = W_down @ h_up
output = h_down + h  # FFN residual

print("Expected:    ", output)
print("Match:", np.allclose(result, output))